In [0]:
dbutils.library.restartPython()

In [0]:
import threading, time
from UTILS import parallel

help(parallel)

In [0]:
##########################################################################################################
# Sample function to print a message at start, delay some time and print a finished message
# --------------------------------------------------------------------------------------------------------
def doSomething(testString = 'default', delay = 3):
    print(f"\t{testString} starting at {time.strftime('%X')}\t\t(Delay = {delay})")
    time.sleep(delay)
    print(f"\t\t{testString} finished at {time.strftime('%X')}")

In [0]:
# A list to hold all the threads we start
threads = []

# Set up some threads
for i in range(4):
    # Create a thread, providing the function to call and the list of parameter values to pass it
    t = threading.Thread(target=doSomething, args=[chr(65 + i), i + 5])

    # Add the thread to our list and start it
    threads.append(t)
    t.start()

# Now all the threads are running...

# Join the thread to the calling thread (this cell) 
# so we stick around to get their finish state
for t in threads:
    t.join()

In [0]:
print("#################\n\tSTART\n#################")

# A list to hold all the threads we start and one to hold the things to do
threads = []
toDo = []

# Number of items and maximum threads to run at a given time
limit = 20
maxDoP = 7

# Set up a bunch of threads 
# (our list of everything that needs doing)
for i in range(limit):
    toDo.append(i)

print(toDo)
# For throttling, we're popping from the list until its empty

# While there are tasks to do...
while len(toDo) > 0:
    # ... and there are threads available
    while len(threads) < maxDoP:
        
        # If we've run out of things to do, break from this loop
        if len(toDo) == 0:
            break
        
        # Get first available item
        i = toDo.pop(0)

        # Create a thread, providing the function to call and the list of parameter values to pass it
        t = threading.Thread(target=doSomething, args=[chr(65 + i), 1 + i % 4])

        # Add the thread to our list and start it
        threads.append(t)
        t.start()
        print(f"+++ {len(threads)} running threads")

    while len(threads) == maxDoP:
        for rt in threads:
            if not rt.is_alive():
                threads.remove(rt)
                print(f"--- {len(threads)} running threads")

# Join the thread to the calling thread (this cell) 
# so we stick around to get their finish state
for t in threads:
    t.join()

print("#################\n\tDONE\n#################")

In [0]:
# Setup our stack of parameter values for various scenarios
paramStack = [  {"nb_path": r"parallel-Test-1", "params": {"testString": "Just a test string"}},
                {"nb_path": r"parallel-Test-1", "timeout_seconds": 30, "params": {"testString": "Specifying timeout_seconds"}},
                {"nb_path": r"parallel-Test-2", "params": {"testString": "Test with short delay", "delaySeconds": 3}},
                {"nb_path": r"parallel-Test-2", "timeout_seconds": 10, "params": {"testString": "!!! Should fail with timeout !!!", "delaySeconds": 20}},
                {"nb_path": r"parallel-Test-2", "params": {"testString": "Test with medium delay", "delaySeconds": 7}},
                {"nb_path": r"parallel-Test-2", "params": {"testString": "Test with long delay", "delaySeconds": 12}}
                ]

# for s in paramStack:
#     for p in s:
#         print(p)

In [0]:
# Timer and messaging
msg = "Run Notebooks in Sequence"
start = time.time()
print(f"===\tStarting: {msg}")

# Simple loop of parameter stack, invoking notebooks in sequence (limit to a few)
for p in paramStack[:2]:
    # Since I'm calling the private function, I have to explicitly supply:
    p['dbutils'] = dbutils
    p['debug'] = True

    # Call the private function (just to demonstrate serial execution)
    parallel._runNotebook(p)

fin = time.time()
print(f"===\tElapsed: {fin - start} seconds")

In [0]:
# Timer and messaging
msg = "Run Notebooks in Parallel"
print(f"===\tStarting: {msg}")

parallel.runParallelNotebooks(dbutils=dbutils, parameterStack=paramStack, maxDoP=4, debug=True)

fin = time.time()
print(f"\n===\tFinished: {msg}")